In [10]:
#Data Understanding
import pandas as pd
import kagglehub
import os

# Download Dataset
print("Downloading dataset...")
folder_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

csv_file_path = os.path.join(folder_path, "IMDB Dataset.csv")

# Load dataset
df = pd.read_csv(csv_file_path)

print("\nDataset Loaded Successfully!")

# Explore dataset
print("\nShape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Positive Review:\n", df[df['sentiment']=='positive']['review'].iloc[0])
print("\nSample Negative Review:\n", df[df['sentiment']=='negative']['review'].iloc[0])


Dataset Loaded Successfully!

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Positive Review:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, 

In [11]:
#2 NLP Preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Cleaning Function
def clean_text(text):
    if pd.isnull(text):
        return ""

    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()   # safer than word_tokenize

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

# Apply cleaning
print("\nCleaning dataset...")
df['cleaned_review'] = df['review'].apply(clean_text)

print("\nSample Cleaned Data:")
print(df[['review', 'cleaned_review']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Cleaning dataset...

Sample Cleaned Data:
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [12]:
#3.Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("\nStarting Feature Engineering...")

# Remove null & empty rows
df = df.dropna(subset=['cleaned_review'])
df = df[df['cleaned_review'].str.strip() != ""]

print("Remaining rows:", len(df))

# Create corpus
corpus = df['cleaned_review'].tolist()

print("Sample cleaned review:", corpus[0])

# Bag of Words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(corpus)
print("BoW Shape:", X_bow.shape)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(corpus)
print("TF-IDF Shape:", X_tfidf.shape)

# Convert labels
df['sentiment_numeric'] = df['sentiment'].map({'positive': 1, 'negative': 0})
y = df['sentiment_numeric'].values


Starting Feature Engineering...
Remaining rows: 50000
Sample cleaned review: one reviewer mentioned watching oz episode youll hooked right exactly happened first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home manyaryans muslim gangsta latino christian italian irish moreso scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show wouldnt dare forget pretty picture painted mainstream audience forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high level graphic violence violence injustice crooked guard wholl sold nickel inmate wholl kil

In [13]:
#4.Model Building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

print("\nSplitting data...")

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

# Models
log_model = LogisticRegression(max_iter=1000)
nb_model = MultinomialNB()
tree_model = DecisionTreeClassifier(random_state=42)

# Train models
print("\nTraining Logistic Regression...")
log_model.fit(X_train, y_train)

print("Training Naive Bayes...")
nb_model.fit(X_train, y_train)

print("Training Decision Tree...")
tree_model.fit(X_train, y_train)

print("\nModels trained successfully!")


Splitting data...
Training size: 40000
Testing size: 10000

Training Logistic Regression...
Training Naive Bayes...
Training Decision Tree...

Models trained successfully!


In [16]:
#5 Model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

print("\nEvaluating models...")

def evaluate_model(y_true, y_pred, name):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4),
        "F1 Score": round(f1_score(y_true, y_pred), 4)
    }

# Predictions
log_pred = log_model.predict(X_test)
nb_pred = nb_model.predict(X_test)
tree_pred = tree_model.predict(X_test)

# Results
results = [
    evaluate_model(y_test, log_pred, "Logistic Regression"),
    evaluate_model(y_test, nb_pred, "Naive Bayes"),
    evaluate_model(y_test, tree_pred, "Decision Tree")
]

comparison_df = pd.DataFrame(results)

print("\nFinal Comparison:\n")
print(comparison_df.to_string(index=False))


Evaluating models...

Final Comparison:

              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8859     0.8774  0.8992    0.8882
        Naive Bayes    0.8534     0.8534  0.8561    0.8548
      Decision Tree    0.7229     0.7286  0.7172    0.7229


In [ ]:
#6 Conclusion & Insights

- TF-IDF performed better than Bag of Words because it considers word importance.
- Logistic Regression achieved the highest accuracy.
- Naive Bayes performed well and was faster.
- Decision Tree showed lower accuracy due to overfitting.
- Preprocessing improved model performance.

Final Best Model: Logistic Regression